# Phase 2 — استرجاع الصور وتجهيز Detection
شغّل الخلايا بالترتيب. هذه المرحلة لا تحتاج GPU. ارفع ملف `Phase1_Results_For_Report.zip` وملف `Phase2_Local_Images.zip` المرفق عند الطلب. تحفظ البيانات على Google Drive ويمكن استكمال التنزيل بعد انقطاع الجلسة.

التقسيمات ثابتة. الصور الموجبة ذات الصناديق الناقصة تُستبعد من بيانات الكشف ويُسجّل سبب الاستبعاد. يلزم فحص الصناديق بصريًا قبل التدريب. لا يعيد هذا الدفتر تدريب التصنيف أو يسترجع أوزانه.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import json, hashlib, zipfile, io, csv, time, math, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
import urllib.request
from PIL import Image, ImageDraw
from IPython.display import display

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/CameraTrap_Phase2')
ROOT.mkdir(parents=True, exist_ok=True)
SOURCE = ROOT/'phase1_source'
SOURCE.mkdir(exist_ok=True)
EXPECTED = 'd3bfb6ba0c9d31412f9d603a6f63ae7c4890f458c5d2aa2a87d7653bb882bb53'
def digest(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def safe_extract(archive, target):
    target = target.resolve()
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            p = (target/item.filename).resolve()
            assert p.is_relative_to(target), 'Unsafe archive path'
            if item.is_dir(): continue
            content = z.read(item)
            p.parent.mkdir(parents=True, exist_ok=True)
            if p.exists(): assert p.read_bytes() == content, f'Existing file differs: {p}'
            else: p.write_bytes(content)
print('Persistent workspace:', ROOT)

## 1 — استرجاع الملفات المرجعية والصور المحلية
أول تشغيل: اختر ملف النتائج وملف الصور المحلية معًا. في التشغيل التالي تُستخدم النسخ المحفوظة.

In [ ]:
needed = []
if not (SOURCE/'handoff/dataset_release.json').exists():
    needed.append('Phase1_Results_For_Report.zip')
if not (ROOT/'local_images/metadata_review.csv').exists():
    needed.append('Phase2_Local_Images.zip')
if needed:
    print('Upload these missing files:', ', '.join(needed))
    uploaded = files.upload()
    for name, content in uploaded.items():
        with zipfile.ZipFile(io.BytesIO(content)) as archive:
            names = set(archive.namelist())
        if 'handoff/dataset_release.json' in names:
            safe_extract(io.BytesIO(content), SOURCE)
            print('Recovered Phase 1 metadata from:', name)
        elif 'metadata_review.csv' in names:
            safe_extract(io.BytesIO(content), ROOT/'local_images')
            print('Recovered local images from:', name)
        else:
            print('Unrecognized archive:', name)
    del uploaded
missing = [str(p) for p in [SOURCE/'handoff/dataset_release.json',
                          ROOT/'local_images/metadata_review.csv'] if not p.is_file()]
if missing:
    raise RuntimeError('Missing inputs: ' + ', '.join(missing) +
                       '. Upload the requested ZIP files and rerun this cell.')
release = json.loads((SOURCE/'handoff/dataset_release.json').read_text())
assert release['schema'] == 2 and release['audit_complete']
for relative, expected in release['files'].items():
    p = (SOURCE/relative).resolve()
    assert p.is_relative_to(SOURCE.resolve()) and digest(p) == expected, relative
manifest = SOURCE/release['data_dir']/'manifest.json'
assert digest(manifest) == EXPECTED
rows = json.loads(manifest.read_text())
assert len({r['id'] for r in rows}) == len(rows)
assert {r['split'] for r in rows} == {'train','val_select','val_calibrate','test'}
for field in ('camera','event','pixel_sha256'):
    groups = {}
    for r in rows:
        assert r.get(field)
        groups.setdefault(r[field], set()).add(r['split'])
    assert all(len(v)==1 for v in groups.values()), field
local = {r['image_id']:r for r in csv.DictReader((ROOT/'local_images/metadata_review.csv').open())}
print('Verified frozen manifest:', len(rows), 'frames')

## 2 — تنزيل الصور والتحقق
يُسجل تطابق الملف والبكسلات مع Phase 1. اختلاف إعادة الترميز يُسجل صراحةً؛ لا يتم تعديل الـmanifest الأصلي. يعتمد إصدار الكشف المشتق على بصمات الصور الجديدة والأبعاد المطابقة. فشل تنزيل أي صورة يمنع تجهيز الإصدار حتى إعادة تشغيل هذه الخلية بنجاح.

In [ ]:
IMAGES = ROOT/'images'
IMAGES.mkdir(exist_ok=True)
BASES = ['https://storage.googleapis.com/public-datasets-lila/caltech-unzipped/cct_images/',
         'https://lilawildlife.blob.core.windows.net/lila-wildlife/caltech-unzipped/cct_images/']
from urllib.parse import quote

def recover(r):
    assert Path(r['id']).name == r['id']
    p = IMAGES/(r['id']+'.jpg')
    try:
        if not p.exists():
            if r['source'] == 'team_collected':
                meta = local[r['id']]
                source = ROOT/'local_images'/meta['filename']
                assert digest(source) == meta['sha256'], 'Local source mismatch'
                raw = source.read_bytes()
            else:
                raw = None
                for attempt in range(3):
                    for base in BASES:
                        try:
                            with urllib.request.urlopen(base+quote(r['file_name']), timeout=45) as response:
                                raw = response.read()
                            break
                        except Exception as exc: error = str(exc)
                    if raw is not None: break
                    time.sleep(attempt+1)
                if raw is None: raise RuntimeError(error)
            with Image.open(io.BytesIO(raw)) as original:
                rgb = original.convert('RGB')
                assert rgb.size == (r['original_width'],r['original_height'])
                rgb.thumbnail((1024,1024),Image.Resampling.LANCZOS)
                tmp = p.with_suffix('.part')
                rgb.save(tmp, format='JPEG', quality=95)
                tmp.replace(p)
        with Image.open(p) as im:
            rgb = im.convert('RGB'); rgb.load()
            assert rgb.size == (r['width'],r['height']), 'Dimension mismatch'
            pixels = hashlib.sha256(rgb.tobytes()).hexdigest()
        return dict(id=r['id'], ok=True, sha256=digest(p),
                    phase1_pixels_match=pixels==r['pixel_sha256'],
                    phase1_file_match=hashlib.md5(p.read_bytes()).hexdigest()==r['processed_file_md5'])
    except Exception as exc:
        return dict(id=r['id'],ok=False,error=str(exc))

recovery = []
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = [pool.submit(recover,r) for r in rows]
    for future in as_completed(futures):
        recovery.append(future.result())
        if len(recovery)%200==0: print(len(recovery),'/',len(rows))
recovery.sort(key=lambda x:x['id'])
(ROOT/'image_recovery.json').write_text(json.dumps(recovery,indent=2))
failed = [r for r in recovery if not r['ok']]
print('Failed:',len(failed),'Pixel differences:',sum(not r.get('phase1_pixels_match',False) for r in recovery if r['ok']))
assert not failed, 'See image_recovery.json; retry this cell after resolving failures.' 

## 3 — تجهيز YOLO وصندوق مراجعة بصري
فئة واحدة: animal. تحفظ val_calibrate مستقلة. الاختبار لا يُستخدم لاختيار العتبات. صور المراجعة التالية من التدريب فقط.

In [ ]:
DATASET = ROOT/'detection'
DATASET.mkdir(exist_ok=True)
included, excluded = [], []
for r in rows:
    if r['animal'] and (r.get('incomplete_boxes',True) or not r['boxes']):
        excluded.append(dict(id=r['id'],split=r['split'],reason='incomplete_positive_boxes'))
        continue
    assert r['animal'] or not r['boxes'], 'Empty frame has boxes'
    lines=[]
    for b in r['boxes']:
        assert b['annotation_source'] in ('human_CCT20','team_human_verified')
        x1,y1,x2,y2=b['xyxy']
        assert all(math.isfinite(v) for v in (x1,y1,x2,y2))
        assert 0<=x1<x2<=1 and 0<=y1<y2<=1
        lines.append(f'0 {(x1+x2)/2:.8f} {(y1+y2)/2:.8f} {x2-x1:.8f} {y2-y1:.8f}')
    image_dir=DATASET/'images'/r['split']; label_dir=DATASET/'labels'/r['split']
    image_dir.mkdir(parents=True,exist_ok=True);label_dir.mkdir(parents=True,exist_ok=True)
    dest=image_dir/(r['id']+'.jpg'); source=IMAGES/dest.name
    if not dest.exists() or digest(dest)!=digest(source): shutil.copyfile(source,dest)
    (label_dir/(r['id']+'.txt')).write_text('\n'.join(lines)+('\n' if lines else ''))
    included.append(r)
(ROOT/'detection_exclusions.json').write_text(json.dumps(excluded,indent=2))
(ROOT/'detection_manifest.json').write_text(json.dumps(included,indent=2))
# JSON is valid YAML; keep calibration separate from model selection.
config=dict(path=str(DATASET),train='images/train',val='images/val_select',test='images/test',names={0:'animal'})
(DATASET/'data.yaml').write_text(json.dumps(config,indent=2))
summary={s:sum(r['split']==s for r in included) for s in ('train','val_select','val_calibrate','test')}
contract=dict(source_manifest_sha256=EXPECTED,counts=summary,excluded=len(excluded),
              image_recovery_sha256=digest(ROOT/'image_recovery.json'),
              manifest_sha256=digest(ROOT/'detection_manifest.json'),
              visual_review_status='pending',scope='Derived detection data; original Phase 1 results unchanged')
(ROOT/'detection_release_candidate.json').write_text(json.dumps(contract,indent=2))
print(summary, 'Excluded:',len(excluded))
import random
preview=random.Random(42).sample([r for r in included if r['split']=='train' and r['boxes']],12)
for r in preview:
    im=Image.open(IMAGES/(r['id']+'.jpg')).convert('RGB');im.thumbnail((640,480))
    draw=ImageDraw.Draw(im)
    for box in r['boxes']:
        x1,y1,x2,y2=box['xyxy'];draw.rectangle((x1*im.width,y1*im.height,x2*im.width,y2*im.height),outline='red',width=2)
    print(r['id']);display(im)
print('Saved to:', ROOT)
print('Review annotations before training. This preview does not establish exhaustive annotation quality.')

## التالي
أرسل `image_recovery.json` و`detection_release_candidate.json` بعد التشغيل. سنراجع اختلافات الصور وتغطية الصناديق قبل اعتماد إصدار التدريب. تبقى بيانات val_calibrate منفصلة للمعايرة. الأقنعة وتدريب الكشف والتقسيم والنشر تأتي في خطوات لاحقة.